In [1]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import netket.experimental as nkx
import sys
sys.path.append('..')
from NES_VMC import NESTotalAnsatz, create_machine,\
    SingleStateAnsatz,create_single_machine,\
        create_machine_matrix,Ham_psi,Ham_Psi,NES_loss_energy,nes_vmc_gradient,\
        NESFermionHopRule,compute_qgt,sampler_info
import optax
from typing import Callable
from functools import partial
from jax.flatten_util import ravel_pytree
import time
import itertools
from pyscf import gto, scf, fci
import numpy as np


jnp.set_printoptions(
    linewidth=9999,
    threshold=jnp.inf,
    precision=8,
    suppress=False,
)


bond_length = 1.8
geometry = [
    ("H", (0.0, 0.0, 0.0)),
    ("H", (bond_length, 0.0, 0.0)),
]

mol = gto.M(atom=geometry, basis="6-31G", verbose=0)
mf = scf.RHF(mol).run(verbose=0)
hf_ground_energy = mf.e_tot

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()

print("=" * 60)
print("H2 / 6-31G 基准")
print("=" * 60)
print(f"HF energy = {hf_ground_energy:.8f} Ha")
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha | excitation = {exc:.4f} eV")

hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=4,
    s=1/2,
    n_fermions_per_spin=(1, 1),
)

K = 2
hi_ext = hi ** K
ha = nkx.operator.from_pyscf_molecule(mol)

SINGLE_SIZE = hi.size

print("=" * 60)
print("Hilbert 信息")
print("=" * 60)
print(f"K = {K}")
print(f"hi.size = {hi.size}")
print(f"hi_ext.size = {hi_ext.size}")
print(f"SINGLE_SIZE = {SINGLE_SIZE}")

target_loss = float(np.sum(E_fcis[:K]))
print(f"target_loss = sum(E_fcis[:K]) = {target_loss:.8f}")


Hatree_Fock = hi.all_states()[0]
alpha_orbs = [0, 1, 2, 3]
beta_orbs  = [4, 5, 6, 7]

single_edges_full = (
    list(itertools.combinations(alpha_orbs, 2))
    + list(itertools.combinations(beta_orbs, 2))
)



ext_edges = []
for k in range(K):
    offset = k * SINGLE_SIZE
    for i, j in single_edges_full:
        ext_edges.append((i + offset, j + offset))

ext_edges = jnp.asarray(ext_edges)

nes_rule = NESFermionHopRule(
    edges=ext_edges,
    K=K,
    single_size=SINGLE_SIZE,
)

print(single_edges_full)

/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: uv is a replacement for pip which helps you follow good software practices.

H2 / 6-31G 基准
HF energy = -0.94605220 Ha
E0 = -1.02613572 Ha | excitation = 0.0000 eV
E1 = -0.97892204 Ha | excitation = 1.2848 eV
E2 = -0.66776157 Ha | excitation = 9.7519 eV
E3 = -0.60817046 Ha | excitation = 11.3734 eV
Hilbert 信息
K = 2
hi.size = 8
hi_ext.size = 16
SINGLE_SIZE = 8
target_loss = sum(E_fcis[:K]) = -2.00505776
[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3), (4, 5), (4, 6), (4, 7), (5, 6), (5, 7), (6, 7)]


In [ ]:
# ============================================================
# 诊断函数 - 从 Stable 版本移植
# ============================================================

def tree_l2_norm(tree):
    leaves = jax.tree_util.tree_leaves(tree)
    if len(leaves) == 0:
        return jnp.array(0.0)
    return jnp.sqrt(
        sum([jnp.sum(jnp.abs(x) ** 2) for x in leaves])
    )


def tree_all_finite(tree):
    leaves = jax.tree_util.tree_leaves(tree)
    if len(leaves) == 0:
        return True
    flags = [jnp.all(jnp.isfinite(x)) for x in leaves]
    return bool(jnp.all(jnp.asarray(flags)))


def unique_ratio_from_samples(samples):
    """
    samples: shape (n_samples, hi_ext.size)
    """
    arr = np.asarray(samples)
    arr = arr.reshape(arr.shape[0], -1)
    unique = len({tuple(row.tolist()) for row in arr})
    return unique / max(arr.shape[0], 1)


def tree_batch_std_norm(tree, batch_size):
    leaves = jax.tree_util.tree_leaves(tree)
    total = 0.0
    for leaf in leaves:
        if leaf.ndim >= 1 and leaf.shape[0] == batch_size:
            centered = leaf - jnp.mean(leaf, axis=0, keepdims=True)
            total = total + jnp.sum(jnp.abs(centered) ** 2)
    return jnp.sqrt(total)


def tree_batch_mean_norm(tree, batch_size):
    leaves = jax.tree_util.tree_leaves(tree)
    total = 0.0
    for leaf in leaves:
        if leaf.ndim >= 1 and leaf.shape[0] == batch_size:
            mean_leaf = jnp.mean(leaf, axis=0)
            total = total + jnp.sum(jnp.abs(mean_leaf) ** 2)
    return jnp.sqrt(total)


def compute_diagnostics_original(
    total_params,
    x_batch,
    samples,
    E_L_mean,
    total_machine,
    total_matrix_machine,
    single_machine_list,
    ha,
    K,
):
    """
    返回当前参数下的完整诊断量 (适配原版非稳定函数)
    """
    n_total = x_batch.shape[0]
    n_diag = min(128, n_total)  # 诊断用的batch大小

    x_diag = x_batch[:n_diag]
    samples_diag = samples[:n_diag]

    # ---------- logΨ 统计 ----------
    log_Psi_batch = jax.vmap(
        lambda xx: total_machine(total_params, xx)
    )(x_batch)

    log_real = jnp.real(log_Psi_batch)
    log_imag = jnp.imag(log_Psi_batch)

    log_real_mean = jnp.mean(log_real)
    log_real_std = jnp.std(log_real)
    log_real_span = jnp.max(log_real) - jnp.min(log_real)

    log_imag_mean = jnp.mean(log_imag)
    log_imag_std = jnp.std(log_imag)
    log_imag_span = jnp.max(log_imag) - jnp.min(log_imag)

    # ---------- Ψ矩阵条件数 ----------
    L_diag = total_matrix_machine(total_params, x_diag)
    Psi_diag = jnp.exp(L_diag)

    try:
        conds = jax.vmap(jnp.linalg.cond)(Psi_diag)
        psi_cond_first = conds[0]
        psi_cond_mean = jnp.mean(conds)
        psi_cond_max = jnp.max(conds)
    except:
        psi_cond_first = jnp.array(0.0)
        psi_cond_mean = jnp.array(0.0)
        psi_cond_max = jnp.array(0.0)

    # ---------- E_L_batch / trace 统计 ----------
    try:
        trace_batch, E_L_batch = NES_loss_energy(
            ha=ha,
            total_matrix_machine=total_matrix_machine,
            single_machine_list=single_machine_list,
            total_params=total_params,
            x=x_diag,
        )
        trace_real = jnp.real(trace_batch)
        trace_imag = jnp.imag(trace_batch)

        trace_real_mean = jnp.mean(trace_real)
        trace_real_std = jnp.std(trace_real)
        trace_real_span = jnp.max(trace_real) - jnp.min(trace_real)

        trace_imag_mean = jnp.mean(trace_imag)
        trace_imag_std = jnp.std(trace_imag)
        trace_imag_span = jnp.max(trace_imag) - jnp.min(trace_imag)
    except:
        trace_real_mean = jnp.array(0.0)
        trace_real_std = jnp.array(0.0)
        trace_real_span = jnp.array(0.0)
        trace_imag_mean = jnp.array(0.0)
        trace_imag_std = jnp.array(0.0)
        trace_imag_span = jnp.array(0.0)
        E_L_batch = None

    # ---------- E_L_mean Hermiticity ----------
    if E_L_mean is not None:
        herm_error = (
            jnp.linalg.norm(E_L_mean - E_L_mean.conj().T)
            / (jnp.linalg.norm(E_L_mean) + 1e-12)
        )
        imag_norm = jnp.linalg.norm(jnp.imag(E_L_mean))
        E_L_herm = 0.5 * (E_L_mean + E_L_mean.conj().T)
        try:
            eig_vals_herm = jnp.linalg.eigvalsh(E_L_herm)
        except:
            eig_vals_herm = jnp.array([0.0, 0.0])
        try:
            eig_vals_raw = jnp.linalg.eigvals(E_L_mean)
            eig_vals_raw = eig_vals_raw[jnp.argsort(jnp.real(eig_vals_raw))]
        except:
            eig_vals_raw = jnp.array([0.0, 0.0])
    else:
        herm_error = jnp.array(0.0)
        imag_norm = jnp.array(0.0)
        eig_vals_herm = jnp.array([0.0, 0.0])
        eig_vals_raw = jnp.array([0.0, 0.0])

    # ---------- sampler unique ratio ----------
    unique_ratio = unique_ratio_from_samples(samples_diag)

    # ---------- dlogΨ batch 方差 ----------
    try:
        grad_logPsi = jax.grad(total_machine, argnums=0, holomorphic=True)
        dlogPsi_batch = jax.vmap(
            grad_logPsi,
            in_axes=(None, 0),
        )(total_params, x_diag)

        dlog_std_norm = tree_batch_std_norm(dlogPsi_batch, n_diag)
        dlog_mean_norm = tree_batch_mean_norm(dlogPsi_batch, n_diag)
        dlog_std_ratio = dlog_std_norm / (dlog_mean_norm + 1e-12)
    except:
        dlog_std_norm = jnp.array(0.0)
        dlog_mean_norm = jnp.array(0.0)
        dlog_std_ratio = jnp.array(0.0)

    return {
        "log_Psi_batch": log_Psi_batch,
        "log_real_mean": log_real_mean,
        "log_real_std": log_real_std,
        "log_real_span": log_real_span,
        "log_imag_mean": log_imag_mean,
        "log_imag_std": log_imag_std,
        "log_imag_span": log_imag_span,
        "psi_cond_first": psi_cond_first,
        "psi_cond_mean": psi_cond_mean,
        "psi_cond_max": psi_cond_max,
        "trace_real_mean": trace_real_mean,
        "trace_real_std": trace_real_std,
        "trace_real_span": trace_real_span,
        "trace_imag_mean": trace_imag_mean,
        "trace_imag_std": trace_imag_std,
        "trace_imag_span": trace_imag_span,
        "herm_error": herm_error,
        "imag_norm": imag_norm,
        "eig_vals_herm": eig_vals_herm,
        "eig_vals_raw": eig_vals_raw,
        "unique_ratio": unique_ratio,
        "dlog_std_norm": dlog_std_norm,
        "dlog_mean_norm": dlog_mean_norm,
        "dlog_std_ratio": dlog_std_ratio,
    }


def log_diagnostics_original(step, loss_mean, grad_norm_raw, grad_norm_update, grad_norm_clipped, diag, target_loss, K, clip_norm=1.0):
    eig_vals_herm = diag["eig_vals_herm"]
    eig_vals_raw = diag["eig_vals_raw"]

    energy_herm_str = " | ".join(
        [f"E{i}_herm={eig_vals_herm[i]:.8f}" for i in range(K)]
    )

    energy_raw_str = " | ".join(
        [f"E{i}_raw={eig_vals_raw[i]:.8f}" for i in range(K)]
    )

    print(f"[Step {step:4d}]")
    print(
        f"Loss={loss_mean:.8f} | "
        f"target={target_loss:.8f} | "
        f"gap={float(loss_mean - target_loss):+.8f}"
    )

    print(
        f"Grad | raw={grad_norm_raw:.4e} | "
        f"update={grad_norm_update:.4e} | "
        f"clipped={grad_norm_clipped:.4e} | "
        f"clip_norm={clip_norm:.2e}"
    )

    print(
        "logΨ.real | "
        f"mean={diag['log_real_mean']:.6f} | "
        f"std={diag['log_real_std']:.4e} | "
        f"span={diag['log_real_span']:.4e}"
    )

    print(
        "logΨ.imag | "
        f"mean={diag['log_imag_mean']:.6f} | "
        f"std={diag['log_imag_std']:.4e} | "
        f"span={diag['log_imag_span']:.4e}"
    )

    print(
        "cond(Ψ) | "
        f"first={diag['psi_cond_first']:.4e} | "
        f"mean={diag['psi_cond_mean']:.4e} | "
        f"max={diag['psi_cond_max']:.4e}"
    )

    print(
        "trace(E_L).real | "
        f"mean={diag['trace_real_mean']:.8f} | "
        f"std={diag['trace_real_std']:.4e} | "
        f"span={diag['trace_real_span']:.4e}"
    )

    print(
        "trace(E_L).imag | "
        f"mean={diag['trace_imag_mean']:.8f} | "
        f"std={diag['trace_imag_std']:.4e} | "
        f"span={diag['trace_imag_span']:.4e}"
    )

    print(
        "E_L_mean diagnostics | "
        f"herm_error={diag['herm_error']:.4e} | "
        f"imag_norm={diag['imag_norm']:.4e}"
    )

    print(
        "sampler / dlogΨ | "
        f"unique_ratio={diag['unique_ratio']:.4f} | "
        f"dlog_std_norm={diag['dlog_std_norm']:.4e} | "
        f"dlog_mean_norm={diag['dlog_mean_norm']:.4e} | "
        f"dlog_std_ratio={diag['dlog_std_ratio']:.4e}"
    )

    print(energy_herm_str)
    print(energy_raw_str)

    # 报警
    if float(diag["log_real_span"]) < 1e-6:
        print(">>> WARNING: logΨ.real span ≈ 0，疑似 amplitude collapse")

    if float(diag["trace_real_std"]) < 1e-8:
        print(">>> WARNING: trace(E_L).real std 很小，covariance 梯度能量项可能无信号")

    if float(diag["dlog_std_ratio"]) < 1e-6:
        print(">>> WARNING: dlogΨ batch variation 很小，参数响应接近常数方向")

    if float(diag["herm_error"]) > 1.0:
        print(">>> WARNING: E_L_mean 非 Hermitian 程度很大，能量诊断不可信")

    print("#" + "-" * 79)

In [4]:
N_CHAINS = 16
N_WARMUP = 100
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 30
N_ITER =400
SINGLE_SIZE = hi.size  # 单个子系统维度 = 4
Natural_Grad = True


total_ansatz = NESTotalAnsatz(hi.size,K,12,rngs=nnx.Rngs(11))
total_machine, total_graphdef,total_params = create_machine(total_ansatz)
total_matrix_machine, total_graphdef,total_params = create_machine_matrix(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)
    
    

optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.sgd(learning_rate=0.01),
)
opt_state = optimizer.init(total_params)

# 诊断参数
PRINT_EVERY = 10
clip_norm = 1.0

nes_sampler = nk.sampler.MetropolisSampler(
    hilbert=hi_ext,
    rule=nes_rule,
    n_chains=16,
    sweep_size=20
)


# 采样器状态初始化（替代原 init_sampler_state）
sampler_rng = jax.random.PRNGKey(21)
sampler_state = nes_sampler.init_state(total_machine, total_params, sampler_rng)

# ==================== 训练循环（带完整诊断） ====================
print("\n" + "="*60)
print("开始多链 NES-VMC 训练 (带完整诊断)")
print("="*60)
print(f"基态能量={E_fcis[0]:.8f} Ha| 第一激发态能量={E_fcis[1]:.8f} Ha| 第二激发态能量={E_fcis[2]:.8f} Ha")
print(f"target_loss = sum(E_fcis[:K]) = {target_loss:.8f}")

history = {
    'step': [],
    'energy_0st': [],
    'energy_1st': [],
    'energy_2st': [],
    'energy_std': [],
    'loss': [],
    'params': [],
    'E_Lmatrix':[],
    'natural_grad':[],
    'grad_flat':[],
    'samples':[],
    'log_Psi':[],
    'log_M':[],
    'log_Psi_mean':[],
    'log_Psi_min':[],
    'log_Psi_max':[],
    'grad_norm':[],
    # 新增诊断历史
    'log_real_std': [],
    'log_real_span': [],
    'trace_real_std': [],
    'trace_real_span': [],
    'dlog_std_norm': [],
    'dlog_mean_norm': [],
    'dlog_std_ratio': [],
    'psi_cond_mean': [],
    'psi_cond_max': [],
    'herm_error': [],
    'imag_norm': [],
    'unique_ratio': [],
}

start_time = time.time()
n_skipped = 0

for step in range(N_ITER):
    # 2. 正式采样
    samples_raw, sampler_state = nes_sampler.sample(
        machine=total_machine, parameters=total_params, 
        state=sampler_state, chain_length=N_SAMPLES_PER_CHAIN
    )
        # 3. 维度重塑，适配梯度函数输入
    samples = samples_raw.reshape(-1, hi_ext.size)
    x_batch = samples.reshape(-1, K, hi.size)
    
    # 3. 计算能量和自然梯度（逻辑和原代码一致）
    grad, loss_mean, E_L_mean = nes_vmc_gradient(ha=ha,
                                                 total_matrix_machine=total_matrix_machine,
                                                 total_machine=total_machine,
                                                 single_machine_list=single_machine_list,
                                                 total_params=total_params,
                                                 x_batch=samples.reshape(-1,K,hi.size))
    #grad = jax.tree_util.tree_map(lambda x: x * 2, grad)
    grad_flat , grad_unravel_fn = ravel_pytree(grad)
    grad_norm_raw = jnp.linalg.norm(grad_flat)
    
    if Natural_Grad == True:
        #grad_flat , grad_unravel_fn = ravel_pytree(grad)
        qgt_reg, unravel_fn = compute_qgt(total_machine, total_params, samples.reshape(-1,K,hi.size), diag_shift=0.1)
        
        # # 自然梯度求解
        natural_grad_flat = jnp.linalg.solve(qgt_reg, grad_flat)
        natural_grad = grad_unravel_fn(natural_grad_flat)
        grad = natural_grad
        grad_norm_update = jnp.linalg.norm(natural_grad_flat)
    else:
        grad_norm_update = grad_norm_raw
        
    grad_norm_clipped = jnp.minimum(grad_norm_update, clip_norm)
    
    # 4. 检查梯度是否有效
    grad_finite = tree_all_finite(grad)
    loss_finite = bool(jnp.isfinite(loss_mean))
    grad_explode = bool(grad_norm_raw > 10.0)  # 梯度爆炸阈值
    
    # 5. 诊断
    need_print = (
        step % PRINT_EVERY == 0
        or step == N_ITER - 1
        or grad_explode
        or (not grad_finite)
        or (not loss_finite)
    )
    
    if need_print:
        diag = compute_diagnostics_original(
            total_params=total_params,
            x_batch=x_batch,
            samples=samples,
            E_L_mean=E_L_mean,
            total_machine=total_machine,
            total_matrix_machine=total_matrix_machine,
            single_machine_list=single_machine_list,
            ha=ha,
            K=K,
        )
        
        log_diagnostics_original(
            step=step,
            loss_mean=loss_mean,
            grad_norm_raw=grad_norm_raw,
            grad_norm_update=grad_norm_update,
            grad_norm_clipped=grad_norm_clipped,
            diag=diag,
            target_loss=target_loss,
            K=K,
            clip_norm=clip_norm,
        )
    
    # 6. 跳过坏更新
    if (not grad_finite) or (not loss_finite) or grad_explode:
        n_skipped += 1
        print(f"【Step {step} Skip】grad_finite={grad_finite} | loss_finite={loss_finite} | grad_norm_raw={float(grad_norm_raw):.6e} | skip_count={n_skipped}")
        continue
    
    # 7. 更新参数
    updates, opt_state = optimizer.update(grad, opt_state, total_params)
    total_params = optax.apply_updates(total_params, updates)
    
    log_Psi_batch = total_machine(total_params, samples.reshape(-1,K,hi.size))
    #eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
    eig_vals, eig_vecs = jnp.linalg.eig(E_L_mean)
    sort_idx = jnp.argsort(eig_vals.real)
    eig_vals, eig_vecs = eig_vals[sort_idx], eig_vecs[:, sort_idx]
    
    grad_norm = jnp.linalg.norm(grad_flat)
    
    history['step'].append(step)
    history['E_Lmatrix'].append(E_L_mean)
    history['samples'].append(samples)
    history['loss'].append(loss_mean)
    history['log_Psi_mean'].append(log_Psi_batch.mean())
    history['log_Psi_min'].append(log_Psi_batch.min())
    history['log_Psi_max'].append(log_Psi_batch.max())
    history['grad_norm'].append(grad_norm)
    history['energy_0st'].append(eig_vals[0])
    history['energy_1st'].append(eig_vals[1])
    history['energy_2st'].append(eig_vals[2])
    history['params'].append(total_params)
    
    # 保存诊断数据
    if need_print and 'diag' in dir():
        history['log_real_std'].append(diag['log_real_std'])
        history['log_real_span'].append(diag['log_real_span'])
        history['trace_real_std'].append(diag['trace_real_std'])
        history['trace_real_span'].append(diag['trace_real_span'])
        history['dlog_std_norm'].append(diag['dlog_std_norm'])
        history['dlog_mean_norm'].append(diag['dlog_mean_norm'])
        history['dlog_std_ratio'].append(diag['dlog_std_ratio'])
        history['psi_cond_mean'].append(diag['psi_cond_mean'])
        history['psi_cond_max'].append(diag['psi_cond_max'])
        history['herm_error'].append(diag['herm_error'])
        history['imag_norm'].append(diag['imag_norm'])
        history['unique_ratio'].append(diag['unique_ratio'])

end_time = time.time()
print(f"\n训练完成!")
print(f"训练耗时：{end_time - start_time:.2f} 秒")
print(f"跳过更新次数：{n_skipped}")


开始多链 NES-VMC 训练 (NetKet 自定义采样器 + 朴素梯度下降)
基态能量=-1.02613572 Ha| 第一激发态能量=-0.97892204 Ha| 第二激发态能量=-0.66776157 Ha
log_Psi: mean=0.155-0.278j | min=-2.927-0.053j | max=1.213+0.308j
grad norm = 1.9902
Step   0 | Loss: 0.5858782313675799|0st能量=-0.02483892-0.00083342j Ha｜1st能量=0.61071715-0.00134566j Ha｜2st能量=0.61071715-0.00134566j Ha
#-----------------------------------------#
log_Psi: mean=2.577+0.425j | min=-0.830-0.180j | max=2.965+2.239j
grad norm = 1.4914
Step  10 | Loss: -1.3557515820171917|0st能量=-0.91728920+0.00552630j Ha｜1st能量=-0.43846238-0.00022329j Ha｜2st能量=-0.43846238-0.00022329j Ha
#-----------------------------------------#
log_Psi: mean=nan+nanj | min=nan+nanj | max=nan+nanj
grad norm = nan
Step  20 | Loss: nan|0st能量=nan+nanj Ha｜1st能量=nan+nanj Ha｜2st能量=nan+nanj Ha
#-----------------------------------------#
log_Psi: mean=nan+nanj | min=nan+nanj | max=nan+nanj
grad norm = nan
Step  30 | Loss: nan|0st能量=nan+nanj Ha｜1st能量=nan+nanj Ha｜2st能量=nan+nanj Ha
#------------------------------

KeyboardInterrupt: 

In [ ]:
Natural_Grad = True

$$
\begin{align*}
\Psi(\mathbf{x})^{-1}\hat{\mathcal{H}}\Psi(\mathbf{x})
&= \mathrm{Tr}\left[ \Psi^{-1}(\mathbf{x})\hat{H}\Psi(\mathbf{x}) \right]
\end{align*}
$$

In [ ]:
import pickle
import os  # 加上这个
# 自动创建 data 文件夹（关键修复）
os.makedirs('./data', exist_ok=True)
if Natural_Grad == True:
    print('保存自然梯度历史记录')
    # 保存 history
    with open('./data/history_natural_gradient_K3.pkl', 'wb') as f:
        pickle.dump(history, f)
else:
    # 保存 history
    with open('./data/history_plain_gradient_K3.pkl', 'wb') as f:
        pickle.dump(history, f)

print("保存成功！")

In [ ]:
import pickle
history_natural= pickle.load(open('./data/history_natural_gradient_K3.pkl', 'rb'))
history_plain= pickle.load(open('./data/history_plain_gradient_K3.pkl', 'rb'))


In [ ]:
history_natural['params'][0]

In [ ]:
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from NES_VMC import E_fcis

# 创建 2行1列 的子图
fig, axs = plt.subplots(3, 3, figsize=(12, 9))
fig.suptitle('Natural Excited State-VMC for $H_2$ K=3 ')
# 第一个子图
axs[0,0].plot(history_natural['energy_0st'],color='orange',label='natural gradient')
axs[0,0].plot(history_plain['energy_0st'],color='blue',label='plain gradient')
axs[0,0].hlines(E_fcis[0],0,len(history_natural['energy_0st']),linestyle='--',color='red')
axs[0,0].set_title('0st Energy')
axs[0,0].set_ylabel('energy')
axs[0,0].set_xlabel('step')
axs[0,0].set_xlabel('step')
axs[0,0].legend()

axs[0,1].plot(history_natural['energy_1st'],color='orange',label='natural gradient')
axs[0,1].plot(history_plain['energy_1st'],color='blue',label='plain gradient')
axs[0,1].hlines(E_fcis[1],0,len(history_natural['energy_1st']),linestyle='--',color='red')
axs[0,1].set_title('1st Energy')
axs[0,1].set_ylabel('energy')
axs[0,1].set_xlabel('step')      
axs[0,1].legend()

axs[0,2].plot(history_natural['energy_2st'],color='orange',label='natural gradient')
axs[0,2].plot(history_plain['energy_2st'],color='blue',label='plain gradient')
axs[0,2].hlines(E_fcis[2],0,len(history_natural['energy_2st']),linestyle='--',color='red')
axs[0,2].set_title('2st Energy')
axs[0,2].set_ylabel('energy')
axs[0,2].set_xlabel('step')      
axs[0,2].legend()


# # 第二个子图
axs[1,0].plot(history_natural['energy_0st']-E_fcis[0],color='orange',label='natural gradient')    
axs[1,0].set_title('0st Energy Error')
axs[1,0].set_xlabel('step')
axs[1,0].set_ylabel('energy')
axs[1,0].legend()



axs[1,1].plot(history_natural['energy_1st']-E_fcis[1],color='orange',label='natural gradient')    
axs[1,1].set_title('1st Energy Error')
axs[1,1].set_xlabel('step')
axs[1,1].set_ylabel('energy')
axs[1,1].legend()

axs[1,2].plot(history_natural['energy_2st']-E_fcis[2],color='orange',label='natural gradient')    
axs[1,2].set_title('2st Energy Error')
axs[1,2].set_xlabel('step')
axs[1,2].set_ylabel('energy')
axs[1,2].legend()


# # 第二个子图
axs[2,0].plot(history_natural['loss'],color='orange',label='natural gradient')  
axs[2,0].plot(history_plain['loss'],color='blue',label='plain gradient')
axs[2,0].set_title('loss')
axs[2,0].set_xlabel('step')
axs[2,0].set_ylabel('loss')
axs[2,0].legend()

axs[2,1].plot(history_natural['grad_norm'],color='orange',label='natural gradient')  
axs[2,1].plot(history_plain['grad_norm'],color='blue',label='plain gradient')
axs[2,1].set_title('grad_norm')
axs[2,1].set_xlabel('step')
axs[2,1].set_ylabel('grad_norm')
axs[2,1].legend()

axs[2,2].plot(history_natural['energy_0st'],color='orange')
axs[2,2].plot(history_natural['energy_1st'],color='orange')
axs[2,2].plot(history_natural['energy_2st'],color='orange')
axs[2,2].hlines(E_fcis[0],0,len(history_natural['energy_0st']),linestyle='--',color='red',label='0st Energy FCI')
axs[2,2].hlines(E_fcis[1],0,len(history_natural['energy_1st']),linestyle='--',color='red',label='1st Energy FCI')
axs[2,2].hlines(E_fcis[2],0,len(history_natural['energy_2st']),linestyle='--',color='red',label='2st Energy FCI')
axs[2,2].set_title('Energy')
axs[2,2].set_xlabel('step')
axs[2,2].set_ylabel('energy')
axs[2,2].legend()

plt.tight_layout()  # 自动调整间距
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from NES_VMC import E_fcis

# 创建 2行1列 的子图
fig, axs = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle('NES-VMC for $H_2$ K=3 ')
# 第一个子图
axs[0].plot(history_natural['energy_0st'],color='orange',label='natural gradient')
axs[0].plot(history_plain['energy_0st'],color='blue',label='plain gradient')
axs[0].hlines(E_fcis[0],0,len(history_natural['energy_0st']),linestyle='--',color='red')
axs[0].set_title('0st Energy')
axs[0].set_ylabel('energy')
axs[0].set_xlabel('step')
axs[0].legend()

axs[1].plot(history_natural['energy_1st'],color='orange',label='natural gradient')
axs[1].plot(history_plain['energy_1st'],color='blue',label='plain gradient')
axs[1].hlines(E_fcis[1],0,len(history_natural['energy_1st']),linestyle='--',color='red')
axs[1].set_title('1st Energy')
axs[1].set_ylabel('energy')
axs[1].set_xlabel('step')      
axs[1].legend()

axs[2].plot(history_natural['energy_2st'],color='orange',label='natural gradient')
axs[2].plot(history_plain['energy_2st'],color='blue',label='plain gradient')
axs[2].hlines(E_fcis[2],0,len(history_natural['energy_2st']),linestyle='--',color='red')
axs[2].set_title('2st Energy')
axs[2].set_ylabel('energy')
axs[2].set_xlabel('step')      
axs[2].legend()


# 第二个子图
axs[3].plot(history_natural['loss'],color='orange',label='natural gradient')            
axs[3].plot(history_plain['loss'],color='blue',label='plain gradient')
axs[3].set_title('loss')
axs[3].set_xlabel('step')
axs[3].set_ylabel('energy')
axs[3].legend()

# 第三个子图
axs[4].plot(history_natural['grad_norm'],color='orange',label='natural gradient')
axs[4].plot(history_plain['grad_norm'],color='blue',label='plain gradient')
axs[4].set_title('grad_norm')
axs[4].set_xlabel('step')
axs[4].set_ylabel('grad_norm')
axs[4].legend()

plt.tight_layout()  # 自动调整间距
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 创建 2行1列 的子图
fig, axs = plt.subplots(1, 5, figsize=(12, 3))
fig.suptitle('NES-VMC for $H_2$ K=3 Natural Gradient Descent')
# 第一个子图
axs[0].plot(history['energy_0st'])
axs[0].hlines(E_fcis[0],0,len(history['energy_0st']),linestyle='--',color='red')
axs[0].set_title('0st Energy')
axs[0].set_ylabel('energy')
axs[0].set_xlabel('step')

axs[1].plot(history['energy_1st'])
axs[1].hlines(E_fcis[1],0,len(history['energy_1st']),linestyle='--',color='red')
axs[1].set_title('1st Energy')
axs[1].set_ylabel('energy')
axs[1].set_xlabel('step')

axs[2].plot(history['energy_2st'])
axs[2].hlines(E_fcis[2],0,len(history['energy_2st']),linestyle='--',color='red')
axs[2].set_title('2st Energy')
axs[2].set_ylabel('energy')
axs[2].set_xlabel('step')

# 第二个子图
axs[3].plot(history['loss'])
axs[3].set_title('loss')
axs[3].set_xlabel('step')
axs[3].set_ylabel('energy')

# 第三个子图
axs[4].plot(history['grad_norm'])
axs[4].set_title('grad_norm')
axs[4].set_xlabel('step')
axs[4].set_ylabel('grad_norm')

plt.tight_layout()  # 自动调整间距
plt.show()

剖析为啥有毛病 

In [ ]:
history['energy_0st'][40:45]

In [ ]:
from collections import Counter
import numpy as np
def sampler_info(samples:jnp.array,K:int):
    test_samples = np.array(samples.reshape(-1, 4*K))
    count = Counter(tuple(each_row.tolist()) for each_row in test_samples)
    for tpl, count_ in count.items():
        print(f"元组 {tpl} 出现了 {count_} 次")
    return count


sampler_info(history['samples'][100],K)

In [ ]:
sampler_info(history['samples'][102],K)

In [ ]:

print(f'我当时保存的: loss: {history["loss"][102]:.3f}|energy_0st: {history["energy_0st"][102]:.3f}|grad_norm: {history["grad_norm"][102]:.3f}')
test_samples = history['samples'][102]
test_params = history['params'][101]
# 3. 计算能量和自然梯度（逻辑和原代码一致）
grad, loss_mean, E_L_mean = nes_vmc_gradient(ha=ha,
                                                total_matrix_machine=total_matrix_machine,
                                                total_machine=total_machine,
                                                single_machine_list=single_machine_list,
                                                total_params=test_params,
                                                x_batch=test_samples.reshape(-1,K,4))

grad_flat , grad_unravel_fn = ravel_pytree(grad)
grad_norm = jnp.linalg.norm(grad_flat)

eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
print(f'我基于当时的 Samples 尝试复现:')
print(f"grad_norm: {grad_norm:.3f}|loss_mean: {loss_mean:.3f}|energy_0st: {eig_vals[0]:.3f}")
print(E_L_mean)

In [ ]:
from NES_VMC import NES_loss_energy

trace, E_L = NES_loss_energy(ha=ha,
                total_matrix_machine=total_matrix_machine,
                single_machine_list=single_machine_list,
                total_params=total_params,
                x=test_samples.reshape(-1,2,4))

trace

In [ ]:
test_samples.reshape(-1,2,4)[0]

In [ ]:
E_L[0]

In [ ]:
def NES_loss_energy(ha, total_matrix_machine,single_machine_list,total_params, x):
    log_M = total_matrix_machine(total_params,x)
    Psi_Matrix = jnp.exp(log_M)
    # 添加正则化项，防止矩阵奇异
    #Psi_Matrix += 1e-6 * jnp.eye(Psi_Matrix.shape[0])
    H_psi_x = Ham_Psi(ha,single_machine_list,total_params,x)
    Psi_Matrix_inv = jnp.linalg.solve(Psi_Matrix, H_psi_x)
    return jnp.real(jnp.trace(Psi_Matrix_inv, axis1=-2, axis2=-1)), Psi_Matrix_inv

In [ ]:
total_matrix_machine(total_params, history['samples'][41].reshape(-1,K,4))